# DermArbiter — Real-Mode End-to-End Colab (O-1)

**Owner:** Emre (Mahmut) — task **O-1** (real model integration)  
**Runtime:** Colab → T4 GPU (Runtime → Change runtime type → T4 GPU)  
**Wall-clock:** ~30-45 min validation + ~5-15 min full e2e pipeline.

## What this notebook does
1. Clone the repo, install deps, confirm T4 + load Colab Secrets.
2. Restore the populated ChromaDB indexes (CaseRAG + GuidelineRAG) from Google Drive.
3. Download PanDerm-ViT-Large weights.
4. Run `scripts/validate_tools_colab.py` — sequentially loads/unloads each of the 9 tools and reports PASS/FAIL + VRAM + latency for each.
5. Mock-mode full pipeline (`run_e2e_gpu.py --mock`) — verifies 5-phase orchestration with no model dependency.
6. **Real-mode full pipeline** — the moment-of-truth uncached run that exercises every tool on a synthetic dermoscopic image and produces a clinical report. **Note:** blocked until Furkan's `model_router._call_local` lands (MedGemma + Qwen agents currently raise `NotImplementedError`); the cell is wired up so flipping the env flag re-runs it.
7. Persist tool-validation JSON + e2e output to Drive.

## Pre-requisites
- Colab Secrets configured: `HF_TOKEN` (Derm1M + MedGemma + DermoGPT access) and `GOOGLE_API_KEY` (Gemini for Specialist/Moderator).
- M-2 already complete: `chroma_cases_derm1m_50k.zip` exists in `MyDrive/dermarbiter/`.
- Optional: `chroma_guidelines.zip` in `MyDrive/dermarbiter/` — if absent, the GuidelineRAG ingest is re-run inline (~10 min).
- **PanDerm weights** — fill in the download URL in Section 3 below (PanDerm authors distribute weights via Google Drive / HF; check https://github.com/SiyuanYan1/PanDerm for the current link).

## VRAM budget on T4 (16 GB total, 4-bit)
| Tool | VRAM | Loaded when |
|---|---|---|
| Qwen3-8B (Skeptic) | ~6 GB | only when Skeptic speaks |
| MedGemma-4B (Generalist) | ~3 GB | only when Generalist speaks |
| PanDerm | ~2 GB | classifier phase only |
| MAKE (CLIP ViT-L-14) | ~1 GB | annotation phase only |
| DermoGPT-RL | ~7 GB | VQA phase only |
| Buffer | ~4 GB | context + activations |

Lazy load + `tool.unload()` between phases keeps it under 16 GB.

## 1. Clone repo + install dependencies + check T4

In [ ]:
REPO_URL = "https://github.com/Furkanahii/DermArbiter.git"
REPO_BRANCH = "emre/derm1m-ingest-and-evaluation"

import os

if not os.path.isdir("/content/DermArbiter"):
    !git clone -b $REPO_BRANCH $REPO_URL /content/DermArbiter
else:
    print("Repo already cloned, fetching latest...")
    !cd /content/DermArbiter && git fetch origin $REPO_BRANCH && git checkout $REPO_BRANCH && git pull

%cd /content/DermArbiter

!pip install -q -e .
!pip install -q huggingface_hub bitsandbytes accelerate

In [ ]:
from google.colab import userdata
import os

for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} loaded from Colab Secrets ({len(os.environ[key])} chars)")
    except userdata.SecretNotFoundError:
        raise RuntimeError(
            f"{key} secret missing. Add it via the 🔑 panel before continuing."
        )

# Mirror so HF picks it up under either name.
os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No CUDA device — set Runtime → Change runtime type → T4 GPU."
print(f"PyTorch CUDA: {torch.version.cuda}  |  Device: {torch.cuda.get_device_name(0)}")

## 2. Restore ChromaDB indexes from Drive

M-2 left `chroma_cases_derm1m_50k.zip` (~200 MB, 50K Derm1M embeddings) in Drive. GuidelineRAG is much smaller (~660 KB, 10 curated entries); if its zip isn't in Drive we just re-run the ingest inline.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path


def _chroma_collection_populated(persist_dir: str, collection_name: str) -> bool:
    """Return True iff persist_dir holds a non-empty named collection.

    Guards against the stale-empty-dir trap: a previous failed ingest can
    leave behind data/chroma_guidelines/chroma.sqlite3 with no collection
    inside it, and a naive `Path.exists()` check then skips the rebuild.
    """
    import chromadb
    if not Path(persist_dir).exists():
        return False
    try:
        client = chromadb.PersistentClient(path=persist_dir)
        cols = {c.name: c for c in client.list_collections()}
        return collection_name in cols and cols[collection_name].count() > 0
    except Exception:
        return False


DRIVE_DIR = Path("/content/drive/MyDrive/dermarbiter")

# --- CaseRAG (required) ---
case_zip = DRIVE_DIR / "chroma_cases_derm1m_50k.zip"
if not case_zip.exists():
    raise FileNotFoundError(
        f"{case_zip} not found. Run notebook 05 (M-2 ingest) first."
    )
if not _chroma_collection_populated("data/chroma_cases", "derm1m_cases"):
    !rm -rf data/chroma_cases
    !mkdir -p data
    !cd data && unzip -q {case_zip}
print(f"✅ CaseRAG restored at data/chroma_cases ({sum(p.stat().st_size for p in Path('data/chroma_cases').rglob('*') if p.is_file()) // 1024 // 1024} MB)")

# --- GuidelineRAG (optional, rebuild if missing OR if collection is empty) ---
guide_zip = DRIVE_DIR / "chroma_guidelines.zip"
if _chroma_collection_populated("data/chroma_guidelines", "clinical_guidelines"):
    print("✅ GuidelineRAG already on disk (populated).")
elif guide_zip.exists():
    !rm -rf data/chroma_guidelines
    !cd data && unzip -q {guide_zip}
    print("✅ GuidelineRAG restored from Drive.")
else:
    print("⚠️  GuidelineRAG zip not in Drive — rebuilding inline (~5 min)...")
    !rm -rf data/chroma_guidelines   # clear any stale half-built state
    !python scripts/build_guideline_rag_index.py
    # Only cache if the rebuild actually populated the collection.
    if _chroma_collection_populated("data/chroma_guidelines", "clinical_guidelines"):
        !cd data && zip -qr {guide_zip} chroma_guidelines
        print(f"✅ GuidelineRAG built + cached to {guide_zip}")
    else:
        print(
            "❌ GuidelineRAG rebuild FAILED — see the script's error output above. "
            "Common causes: (a) build_guideline_rag_index.py missing from the cloned "
            "repo branch, (b) data/guideline_seed/guidelines.jsonl missing, (c) HF "
            "sentence-transformers download blocked. Fix and re-run this cell."
        )

import chromadb
for path, name in [("data/chroma_cases", "derm1m_cases"),
                    ("data/chroma_guidelines", "clinical_guidelines")]:
    if not Path(path).exists():
        print(f"  ⏭️ {path} not on disk — skipping count check.")
        continue
    client = chromadb.PersistentClient(path=path)
    cols = {c.name: c for c in client.list_collections()}
    if name in cols:
        print(f"  {path}::{name}  count = {cols[name].count():,}")
    else:
        print(f"  ⚠️ {path}::{name} missing (collections found: {list(cols)})")

## 3. Download PanDerm weights

PanDerm-ViT-Large encoder (~1.2 GB) + classification head (~10 MB). The authors distribute the weights via Google Drive / HuggingFace — check https://github.com/SiyuanYan1/PanDerm for the current download instructions, then fill in the URLs below.

If you have a Google Drive share link: `pip install gdown` then `gdown --id <FILE_ID> -O weights/panderm.pth`.  
If there's an HF mirror: `huggingface-cli download <repo_id> panderm.pth --local-dir weights/`.

In [ ]:
import os
from pathlib import Path

PANDERM_WEIGHT_URL = os.environ.get("PANDERM_WEIGHT_URL", "")  # TODO: fill in or set env var
PANDERM_HEAD_URL = os.environ.get("PANDERM_HEAD_URL", "")      # TODO: fill in or set env var

weights_dir = Path("weights"); weights_dir.mkdir(exist_ok=True)

if (weights_dir / "panderm.pth").exists() and (weights_dir / "panderm_head.pth").exists():
    print("✅ PanDerm weights already present.")
elif PANDERM_WEIGHT_URL and PANDERM_HEAD_URL:
    !wget -q -O weights/panderm.pth "$PANDERM_WEIGHT_URL"
    !wget -q -O weights/panderm_head.pth "$PANDERM_HEAD_URL"
    print("✅ PanDerm weights downloaded.")
else:
    print(
        "⚠️  PANDERM_WEIGHT_URL / PANDERM_HEAD_URL not set. "
        "PanDerm test will be reported as FAIL by validate_tools_colab.py; "
        "the remaining 8 tools still run.\n"
        "   Get the weights from https://github.com/SiyuanYan1/PanDerm and "
        "either set the env vars above or download into weights/panderm.pth + weights/panderm_head.pth manually."
    )

!ls -lh weights/ 2>/dev/null || true

## 4. Validate all 9 tools individually

`scripts/validate_tools_colab.py` creates a synthetic dermoscopic image, then sequentially loads each tool, runs `tool.run(image_path=...)`, and unloads it before the next. The script prints a per-tool PASS/FAIL table plus VRAM peak and writes `results/tool_validation.json` for the next cell to summarise.

In [ ]:
import time
t0 = time.time()
!python scripts/validate_tools_colab.py
print(f"\nValidation wall-clock: {(time.time() - t0)/60:.1f} min")

In [ ]:
import json
from pathlib import Path

report_path = Path("results/tool_validation.json")
if not report_path.exists():
    print("⚠️ tool_validation.json not produced — see the prior cell's output for errors.")
else:
    data = json.loads(report_path.read_text())
    results = data.get("results", data) if isinstance(data, dict) else data
    print(f"{'Tool':<28} {'Status':<8} {'Elapsed':>10} {'VRAM':>8}  Notes")
    print("-" * 90)
    for r in results:
        status = r.get("status", "?")
        emoji = {"PASS": "✅", "FAIL": "❌", "SKIPPED": "⏭️", "ERROR": "💥"}.get(status, "?")
        print(
            f"{r.get('tool_name','?'):<28} {emoji} {status:<5} {r.get('elapsed_ms', 0):>8.0f}ms"
            f" {r.get('vram_peak_gb', 0):>6.1f}GB  {r.get('notes', r.get('error_msg', ''))[:60]}"
        )
    passes = sum(1 for r in results if r.get("status") == "PASS")
    print(f"\n  {passes}/{len(results)} tools passed.")

## 5. Full pipeline — mock first (sanity check)

Runs the 5-phase orchestrator with mock agents/tools to confirm phase wiring, blackboard state, and report rendering all work end-to-end. Should finish in <5 s.

In [ ]:
!python scripts/run_e2e_gpu.py \
    --mock \
    --query "Changing mole on back, asymmetric border" \
    --age 45 --sex male --fitzpatrick 3 \
    --output-json results/e2e_mock.json

## 6. Full pipeline — REAL mode

This is the **O-1 acceptance test**. Real Gemini (Specialist + Moderator), real MedGemma (Generalist), real Qwen3 (Skeptic), real 9 tools, real ChromaDB. Wall-clock ~5-15 min on T4 (dominated by lazy model loads, Gemini API round-trips, and sequential tool runs).

> ⚠️ **Currently blocked on Furkan's F5:** `model_router._call_local` raises `NotImplementedError` for `local_hf` and `groq_api` backends — i.e., the Generalist (MedGemma) and Skeptic (Qwen3) agents will crash. Until that lands the cell is left here as the test harness; flipping `BLOCKED_ON_FURKAN = False` once F5 is merged is the only change needed.

In [ ]:
BLOCKED_ON_FURKAN = True  # flip to False once Furkan's _call_local lands

if BLOCKED_ON_FURKAN:
    print(
        "⏭️  Skipping real-mode e2e run. Until model_router._call_local is implemented, "
        "the Generalist (MedGemma) and Skeptic (Qwen3) agents will raise NotImplementedError. "
        "Flip BLOCKED_ON_FURKAN to False once F5 is merged."
    )
else:
    !python scripts/run_e2e_gpu.py \
        --query "Changing mole on back, asymmetric border" \
        --image data/test_synthetic_lesion.jpg \
        --age 45 --sex male --fitzpatrick 3 \
        --output-json results/e2e_real.json \
        --log-level INFO

## 7. Persist results to Drive

Saves the tool-validation JSON and e2e outputs so the run survives session disconnect. Re-running this notebook with the same flags is idempotent: existing files in Drive are overwritten.

In [ ]:
from pathlib import Path
import shutil

DRIVE_DIR = Path("/content/drive/MyDrive/dermarbiter")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for src in ["results/tool_validation.json", "results/e2e_mock.json", "results/e2e_real.json"]:
    src_path = Path(src)
    if src_path.exists():
        dst = DRIVE_DIR / src_path.name
        shutil.copy(src_path, dst)
        print(f"  ✅ {src} → {dst}")
    else:
        print(f"  ⏭️ {src} not produced (skipping)")

!ls -lh /content/drive/MyDrive/dermarbiter/*.json 2>/dev/null

## Next steps

- **If all 9 tools PASSed** (Section 4): O-1 Emre-side is complete. Wait on Furkan's `_call_local`, flip the BLOCKED flag in Section 6, re-run, and we're at **O-1 acceptance**.
- **If a tool FAILed**: open `results/tool_validation.json` for the full `error_msg`. Most common causes:
  - PanDerm → weights not downloaded (Section 3).
  - DermoGPT / MedGemma → HF token doesn't have access to the gated repos. Re-accept the usage agreement on huggingface.co.
  - CaseRAG / GuidelineRAG → ChromaDB unzip didn't land at the expected path.
- **Once O-1 passes**: proceed to **O-2** (benchmark runs on DermAgent 642-image subset → HAM10000 test split → Fitzpatrick17k fairness).